# ARES Terrain Segmentation Baseline v0.1

This notebook trains a small PyTorch U-Net on a bounded, deterministic sample streamed from the `hassanjbara/AI4MARS` Hugging Face training split. It is intended for the Google Colab NVIDIA T4 runtime only and does not represent full-dataset performance.

## Scope and safeguards

- No package installation or pretrained/external model downloads are performed.
- At most 2,000 raw streamed records are inspected to collect 640 usable labeled samples.
- The 512/128 split is a preliminary image-level split because rover sequence metadata has not been verified.
- Dataset samples and checkpoints are not stored in the Git repository. The best checkpoint is written to Google Drive.
- Class IDs are preserved as `0=soil`, `1=bedrock`, `2=sand`, `3=big rock`, and `255=ignore/null`.

## 1. Runtime and dependency verification

In [54]:
import importlib.util
import os
import platform
import sys

REQUIRED_PACKAGES = ["torch", "numpy", "PIL", "matplotlib", "datasets"]
missing_dependencies = [
    package for package in REQUIRED_PACKAGES
    if importlib.util.find_spec(package) is None
]
try:
    import google.colab  # noqa: F401
    running_in_colab = True
except ImportError:
    running_in_colab = False

print({
    "running_in_colab": running_in_colab,
    "python": sys.version.split()[0],
    "hostname": platform.node(),
    "missing_dependencies": missing_dependencies,
    "COLAB_GPU": os.environ.get("COLAB_GPU", "not set"),
})
if missing_dependencies:
    raise RuntimeError(f"Missing dependencies; install them explicitly before continuing: {missing_dependencies}")

{'running_in_colab': True, 'python': '3.13.16', 'hostname': 'ad6fb4e1edb1', 'missing_dependencies': [], 'COLAB_GPU': '1'}


This cell reports missing dependencies instead of installing them. A missing package is a runtime prerequisite, not a reason to silently change the environment.

In [55]:
import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
gpu_name = torch.cuda.get_device_name(0) if torch.cuda.is_available() else None
print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("GPU:", gpu_name)
if torch.cuda.is_available():
    print(torch.cuda.get_device_properties(0))
if not torch.cuda.is_available() or gpu_name != "Tesla T4":
    raise RuntimeError(f"Expected an NVIDIA Tesla T4 CUDA runtime; found CUDA={torch.cuda.is_available()}, GPU={gpu_name}")

PyTorch: 2.11.0+cu130
CUDA available: True
GPU: Tesla T4
_CudaDeviceProperties(name='Tesla T4', major=7, minor=5, total_memory=14912MB, multi_processor_count=40, uuid=168aaae0-40f3-020e-49fc-257762ec1ee1, pci_bus_id=0, pci_device_id=4, pci_domain_id=0, L2_cache_size=4MB)


Training is intentionally gated on a detected NVIDIA Tesla T4. No CPU fallback is used for this baseline because the project requires training in Colab.

## 2. Reproducibility configuration

In [56]:
import json
import random
import numpy as np

SEED = 2026
DATASET_ID = "hassanjbara/AI4MARS"
DATASET_SPLIT = "train"
MAX_RAW_RECORDS = 2_000
TARGET_SAMPLES = 640
TRAIN_SAMPLES = 512
VALIDATION_SAMPLES = 128
IMAGE_SIZE = (256, 256)
NUM_CLASSES = 4
IGNORE_INDEX = 255
BATCH_SIZE = 8
EPOCHS = 5
LEARNING_RATE = 0.001
RESUME_CHECKPOINT = False
CHECKPOINT_PATH = "/content/drive/MyDrive/ARES/checkpoints/terrain_unet_best.pt"
assert TRAIN_SAMPLES + VALIDATION_SAMPLES == TARGET_SAMPLES
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
print(json.dumps({
    "seed": SEED,
    "dataset": f"{DATASET_ID}:{DATASET_SPLIT}",
    "max_raw_records": MAX_RAW_RECORDS,
    "target_samples": TARGET_SAMPLES,
    "train_samples": TRAIN_SAMPLES,
    "validation_samples": VALIDATION_SAMPLES,
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "epochs": EPOCHS,
    "learning_rate": LEARNING_RATE,
    "device": str(DEVICE),
}, indent=2))

{
  "seed": 2026,
  "dataset": "hassanjbara/AI4MARS:train",
  "max_raw_records": 2000,
  "target_samples": 640,
  "train_samples": 512,
  "validation_samples": 128,
  "image_size": [
    256,
    256
  ],
  "batch_size": 8,
  "epochs": 5,
  "learning_rate": 0.001,
  "device": "cuda"
}


All sampling and training constants are centralized here. Re-running this cell resets the random generators and preserves the same bounded sampling configuration.

## 3. Bounded deterministic streaming dataset collection

The streaming dataset is shuffled with a fixed seed and bounded buffer. The loop stops after 2,000 raw records, skips missing pairs, `has_labels=False`, and all-null masks, and stores only resized tensors for the 640 selected examples.

In [62]:
import io
import numpy as np
import torch

from PIL import Image, UnidentifiedImageError
from datasets import load_dataset, Image as HFImage


# ============================================================
# 1. EXPLICIT IMAGE DECODING
# ============================================================

def decode_image(value):
    """
    Decode an image without relying on Hugging Face's
    automatic PIL image decoder.

    Supported representations:
    - PIL Image
    - bytes / bytearray / memoryview
    - {"bytes": ..., "path": ...}
    - file path string
    """

    if value is None:
        raise ValueError("Image value is None.")

    if isinstance(value, Image.Image):
        return value.copy()

    if isinstance(value, dict):

        if value.get("bytes") is not None:
            value = value["bytes"]

        elif value.get("path") is not None:
            value = value["path"]

        else:
            raise ValueError(
                "Image mapping contains neither bytes nor path."
            )

    if isinstance(value, (bytes, bytearray, memoryview)):
        image = Image.open(io.BytesIO(bytes(value)))

    elif isinstance(value, str):
        image = Image.open(value)

    else:
        raise TypeError(
            f"Unsupported image representation: {type(value)}"
        )

    image.load()

    return image.copy()


# ============================================================
# 2. IMAGE AND MASK PREPROCESSING
# ============================================================

def prepare_pair(image, mask):

    image = image.convert("L").resize(
        IMAGE_SIZE[::-1],
        Image.Resampling.BILINEAR
    )

    mask = mask.convert("L").resize(
        IMAGE_SIZE[::-1],
        Image.Resampling.NEAREST
    )

    image_tensor = torch.from_numpy(
        np.asarray(image, dtype=np.float32) / 255.0
    ).unsqueeze(0)

    mask_tensor = torch.from_numpy(
        np.asarray(mask, dtype=np.uint8).astype(np.int64)
    )

    return image_tensor, mask_tensor


# ============================================================
# 3. LOAD STREAMING DATASET
# ============================================================

stream = load_dataset(
    DATASET_ID,
    split=DATASET_SPLIT,
    streaming=True
)

# Disable automatic decoding for both image columns.
stream = stream.cast_column(
    "image",
    HFImage(decode=False)
)

stream = stream.cast_column(
    "label_mask",
    HFImage(decode=False)
)

# Deterministic shuffle.
stream = stream.shuffle(
    seed=SEED,
    buffer_size=MAX_RAW_RECORDS
)


# ============================================================
# 4. VERIFY FEATURE DECODING CONFIGURATION
# ============================================================

image_feature = stream.features["image"]
mask_feature = stream.features["label_mask"]

print("Image feature:", image_feature)
print("Mask feature:", mask_feature)

print("Image decoding enabled:", image_feature.decode)
print("Mask decoding enabled:", mask_feature.decode)

assert image_feature.decode is False, (
    "Image automatic decoding is still enabled."
)

assert mask_feature.decode is False, (
    "Mask automatic decoding is still enabled."
)

print("Feature decoding checks passed.")


# ============================================================
# 5. INITIALIZE COLLECTION
# ============================================================

selected_images = []
selected_masks = []
selected_records = []

raw_records_inspected = 0

skipped_missing_pair = 0
skipped_without_labels = 0
skipped_all_null = 0

skipped_corrupt_image = 0
skipped_corrupt_mask = 0


# ============================================================
# 6. STREAM AND COLLECT SAMPLES
# ============================================================

for raw_index, record in enumerate(stream):

    if raw_records_inspected >= MAX_RAW_RECORDS:
        break

    if len(selected_images) >= TARGET_SAMPLES:
        break

    raw_records_inspected += 1

    image_raw = record.get("image")
    mask_raw = record.get("label_mask")

    # --------------------------------------------------------
    # Missing image or mask
    # --------------------------------------------------------

    if image_raw is None or mask_raw is None:
        skipped_missing_pair += 1
        continue

    # --------------------------------------------------------
    # Require has_labels=True
    # --------------------------------------------------------

    if record.get("has_labels") is not True:
        skipped_without_labels += 1
        continue

    # --------------------------------------------------------
    # Decode image
    # --------------------------------------------------------

    try:

        image = decode_image(image_raw)

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        TypeError
    ):

        skipped_corrupt_image += 1
        continue

    # --------------------------------------------------------
    # Decode mask
    # --------------------------------------------------------

    try:

        mask = decode_image(mask_raw)

    except (
        UnidentifiedImageError,
        OSError,
        ValueError,
        TypeError
    ):

        skipped_corrupt_mask += 1
        continue

    # --------------------------------------------------------
    # Convert mask to grayscale
    # --------------------------------------------------------

    mask = mask.convert("L")

    mask_array = np.asarray(mask)

    # --------------------------------------------------------
    # Validate semantic IDs
    # --------------------------------------------------------

    observed_ids = set(
        np.unique(mask_array).tolist()
    )

    allowed_ids = {
        0,
        1,
        2,
        3,
        IGNORE_INDEX
    }

    unexpected_ids = observed_ids - allowed_ids

    if unexpected_ids:

        raise ValueError(
            f"Unexpected mask IDs at raw record "
            f"{raw_index}: {sorted(unexpected_ids)}"
        )

    # --------------------------------------------------------
    # Skip all-null masks
    # --------------------------------------------------------

    if not np.isin(
        mask_array,
        [0, 1, 2, 3]
    ).any():

        skipped_all_null += 1
        continue

    # --------------------------------------------------------
    # Prepare tensors
    # --------------------------------------------------------

    image_size_before_resize = tuple(image.size)
    mask_size_before_resize = tuple(mask.size)

    image_tensor, mask_tensor = prepare_pair(
        image,
        mask
    )

    selected_images.append(image_tensor)
    selected_masks.append(mask_tensor)

    selected_records.append({

        "raw_index": raw_index,

        "has_labels": record.get("has_labels"),

        "image_size_before_resize": image_size_before_resize,

        "mask_size_before_resize": mask_size_before_resize,

        "mask_ids_before_resize": sorted(observed_ids)

    })


# ============================================================
# 7. COLLECTION VALIDATION
# ============================================================

usable_samples = len(selected_images)

if usable_samples < TARGET_SAMPLES:

    raise RuntimeError(

        f"Insufficient usable samples: "
        f"{usable_samples}/{TARGET_SAMPLES}.\n"

        f"Raw records inspected: {raw_records_inspected}\n"

        f"Missing pairs: {skipped_missing_pair}\n"

        f"Without labels: {skipped_without_labels}\n"

        f"All-null masks: {skipped_all_null}\n"

        f"Corrupt images: {skipped_corrupt_image}\n"

        f"Corrupt masks: {skipped_corrupt_mask}"

    )


# ============================================================
# 8. CREATE FINAL TENSORS
# ============================================================

selected_images = torch.stack(selected_images)

selected_masks = torch.stack(selected_masks)


# ============================================================
# 9. VERIFY TENSOR SHAPES
# ============================================================

assert selected_images.shape == (
    TARGET_SAMPLES,
    1,
    IMAGE_SIZE[0],
    IMAGE_SIZE[1]
)

assert selected_masks.shape == (
    TARGET_SAMPLES,
    IMAGE_SIZE[0],
    IMAGE_SIZE[1]
)


# ============================================================
# 10. COLLECTION REPORT
# ============================================================

collection_report = {

    "dataset": f"{DATASET_ID}:{DATASET_SPLIT}",

    "raw_records_inspected": raw_records_inspected,

    "usable_samples": usable_samples,

    "skipped_missing_pair": skipped_missing_pair,

    "skipped_without_labels": skipped_without_labels,

    "skipped_all_null": skipped_all_null,

    "skipped_corrupt_image": skipped_corrupt_image,

    "skipped_corrupt_mask": skipped_corrupt_mask,

    "image_tensor_shape": tuple(selected_images.shape),

    "mask_tensor_shape": tuple(selected_masks.shape),

    "device": str(DEVICE)

}

print("\n========== DATASET COLLECTION REPORT ==========")

for key, value in collection_report.items():

    print(f"{key}: {value}")

print("\nDataset collection completed successfully.")

Image feature: Image(mode=None, decode=False)
Mask feature: Image(mode=None, decode=False)
Image decoding enabled: False
Mask decoding enabled: False
Feature decoding checks passed.

========== DATASET COLLECTION REPORT ==========
dataset: hassanjbara/AI4MARS:train
raw_records_inspected: 757
usable_samples: 640
skipped_missing_pair: 112
skipped_without_labels: 0
skipped_all_null: 5
skipped_corrupt_image: 0
skipped_corrupt_mask: 0
image_tensor_shape: (640, 1, 256, 256)
mask_tensor_shape: (640, 256, 256)
device: cuda

Dataset collection completed successfully.


The selected tensors are a bounded in-memory subset, not a dataset cache. The split is image-level and preliminary: sequence metadata has not been verified, so validation performance may be optimistic if related frames occur in both subsets.

In [ ]:
from torch.utils.data import DataLoader, TensorDataset

train_images = selected_images[:TRAIN_SAMPLES]
train_masks = selected_masks[:TRAIN_SAMPLES]
validation_images = selected_images[TRAIN_SAMPLES:TARGET_SAMPLES]
validation_masks = selected_masks[TRAIN_SAMPLES:TARGET_SAMPLES]
assert train_images.shape[0] == TRAIN_SAMPLES
assert validation_images.shape[0] == VALIDATION_SAMPLES
train_loader = DataLoader(TensorDataset(train_images, train_masks), batch_size=BATCH_SIZE, shuffle=True, generator=torch.Generator().manual_seed(SEED))
validation_loader = DataLoader(TensorDataset(validation_images, validation_masks), batch_size=BATCH_SIZE, shuffle=False)
print("train shapes:", tuple(train_images.shape), tuple(train_masks.shape))
print("validation shapes:", tuple(validation_images.shape), tuple(validation_masks.shape))

## 3a. Compact data execution audit

This report contains scalar counts only. It is written to `/content/ares_data_audit.json`, outside the Git repository, and does not save images, masks, or tensors.

In [ ]:
from pathlib import Path

class_pixel_counts = torch.bincount(
    selected_masks.reshape(-1),
    minlength=IGNORE_INDEX + 1,
)[:NUM_CLASSES].tolist()
audit_report = {
    "dataset": f"{DATASET_ID}:{DATASET_SPLIT}",
    "raw_records_inspected": int(raw_records_inspected),
    "usable_samples": int(selected_images.shape[0]),
    "skipped_records": {
        "missing_image_or_mask": int(skipped_missing_pair),
        "without_labels": int(skipped_without_labels),
        "all_null_mask": int(skipped_all_null),
    },
    "skipped_records_total": int(
        skipped_missing_pair + skipped_without_labels + skipped_all_null
    ),
    "class_distribution_pixels": {
        str(class_id): int(count)
        for class_id, count in enumerate(class_pixel_counts)
    },
    "train_samples": int(train_images.shape[0]),
    "validation_samples": int(validation_images.shape[0]),
}
audit_path = Path("/content/ares_data_audit.json")
audit_path.write_text(json.dumps(audit_report, indent=2) + "\n", encoding="utf-8")
print(json.dumps(audit_report, separators=(",", ":")))
print(f"Saved compact audit report to {audit_path}")

The audit confirms the bounded collection result and split sizes using counts derived from the selected tensors. The JSON artifact is intentionally stored in Colab's `/content` area rather than the repository.

The first 512 selected usable samples form training data and the remaining 128 form validation data. No test claim is made; this is only a preliminary baseline split.

## 4. Lightweight U-Net

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

class DoubleConv(nn.Module):
    def __init__(self, in_channels, out_channels):
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),
        )

    def forward(self, x):
        return self.block(x)

class TinyUNet(nn.Module):
    def __init__(self, num_classes=NUM_CLASSES):
        super().__init__()
        self.enc1 = DoubleConv(1, 16)
        self.enc2 = DoubleConv(16, 32)
        self.enc3 = DoubleConv(32, 64)
        self.bottleneck = DoubleConv(64, 128)
        self.pool = nn.MaxPool2d(2)
        self.up3 = nn.ConvTranspose2d(128, 64, 2, stride=2)
        self.dec3 = DoubleConv(128, 64)
        self.up2 = nn.ConvTranspose2d(64, 32, 2, stride=2)
        self.dec2 = DoubleConv(64, 32)
        self.up1 = nn.ConvTranspose2d(32, 16, 2, stride=2)
        self.dec1 = DoubleConv(32, 16)
        self.head = nn.Conv2d(16, num_classes, 1)
    def forward(self, x):
        e1 = self.enc1(x)
        e2 = self.enc2(self.pool(e1))
        e3 = self.enc3(self.pool(e2))
        b = self.bottleneck(self.pool(e3))
        d3 = self.dec3(torch.cat([self.up3(b), e3], dim=1))
        d2 = self.dec2(torch.cat([self.up2(d3), e2], dim=1))
        d1 = self.dec1(torch.cat([self.up1(d2), e1], dim=1))
        return self.head(d1)
model = TinyUNet().to(DEVICE)
parameter_count = sum(parameter.numel() for parameter in model.parameters())
print("parameters:", parameter_count)
assert model(torch.zeros(1, 1, *IMAGE_SIZE, device=DEVICE)).shape == (1, NUM_CLASSES, *IMAGE_SIZE)

The baseline uses one grayscale input channel and four logits per pixel. It has no pretrained weights or external model dependency.

In [ ]:
train_pixel_counts = torch.bincount(train_masks.reshape(-1), minlength=256)[:NUM_CLASSES].float()
total_valid_train_pixels = train_pixel_counts.sum()
if total_valid_train_pixels.item() == 0:
    raise RuntimeError("Training masks contain no valid semantic pixels.")
class_weights = total_valid_train_pixels / (NUM_CLASSES * train_pixel_counts.clamp_min(1.0))
class_weights = class_weights.to(DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, ignore_index=IGNORE_INDEX)
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
print("training pixel counts:", train_pixel_counts.tolist())
print("class weights:", class_weights.detach().cpu().tolist())

Class weights are computed only from training pixels with IDs 0–3. ID 255 is excluded from the histogram and ignored by both the loss and later metrics.

## 5. Training and resumable Drive checkpointing

In [ ]:
from pathlib import Path

if not running_in_colab:
    raise RuntimeError("Checkpointing is restricted to Google Drive in the Colab runtime.")
from google.colab import drive
drive.mount("/content/drive", force_remount=False)
Path(CHECKPOINT_PATH).parent.mkdir(parents=True, exist_ok=True)
print("checkpoint path:", CHECKPOINT_PATH)


The checkpoint directory is outside the Git repository. Drive mounting is explicit and occurs only when this cell is run in Colab.

In [ ]:
import time

history = {"train_loss": [], "validation_loss": []}
start_epoch = 0
best_validation_loss = float("inf")
if RESUME_CHECKPOINT and Path(CHECKPOINT_PATH).exists():
    checkpoint = torch.load(CHECKPOINT_PATH, map_location=DEVICE, weights_only=False)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    start_epoch = int(checkpoint["epoch"]) + 1
    best_validation_loss = float(checkpoint["best_validation_loss"])
    history = checkpoint.get("history", history)
    print("resumed from epoch:", start_epoch)
for epoch in range(start_epoch, EPOCHS):
    model.train()
    train_loss_sum = 0.0
    train_batches = 0
    for images, masks in train_loader:
        valid_pixels = (masks != IGNORE_INDEX).sum().item()
        if valid_pixels == 0:
            continue
        images, masks = images.to(DEVICE), masks.to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(images), masks)
        if not torch.isfinite(loss):
            raise RuntimeError(f"Non-finite training loss at epoch {epoch + 1}")
        loss.backward()
        optimizer.step()
        train_loss_sum += loss.item()
        train_batches += 1
    model.eval()
    validation_loss_sum = 0.0
    validation_batches = 0
    with torch.no_grad():
        for images, masks in validation_loader:
            if (masks != IGNORE_INDEX).sum().item() == 0:
                continue
            loss = criterion(model(images.to(DEVICE)), masks.to(DEVICE))
            validation_loss_sum += loss.item()
            validation_batches += 1
    if train_batches == 0 or validation_batches == 0:
        raise RuntimeError("An epoch had no valid supervised batches.")
    train_loss = train_loss_sum / train_batches
    validation_loss = validation_loss_sum / validation_batches
    history["train_loss"].append(train_loss)
    history["validation_loss"].append(validation_loss)
    print(f"epoch {epoch + 1}/{EPOCHS}: train_loss={train_loss:.4f}, validation_loss={validation_loss:.4f}")
    if validation_loss < best_validation_loss:
        best_validation_loss = validation_loss
        torch.save({
            "epoch": epoch,
            "model_state_dict": model.state_dict(),
            "optimizer_state_dict": optimizer.state_dict(),
            "best_validation_loss": best_validation_loss,
            "history": history,
            "configuration": {
                "seed": SEED,
                "dataset_id": DATASET_ID,
                "split": DATASET_SPLIT,
                "target_samples": TARGET_SAMPLES,
                "train_samples": TRAIN_SAMPLES,
                "validation_samples": VALIDATION_SAMPLES,
                "image_size": IMAGE_SIZE,
                "batch_size": BATCH_SIZE,
                "epochs": EPOCHS,
                "learning_rate": LEARNING_RATE,
                "ignore_index": IGNORE_INDEX,
                "device": str(DEVICE),
            },
        }, CHECKPOINT_PATH)
        print("saved best checkpoint:", CHECKPOINT_PATH)


Training records mean loss per valid batch and saves only the best validation checkpoint. Empty/all-ignore batches are skipped, while an epoch with no valid batches fails explicitly.

## 6. Evaluation on the preliminary validation subset

In [ ]:
def update_confusion_matrix(confusion, predictions, targets):
    valid = targets != IGNORE_INDEX
    targets = targets[valid].to(torch.int64)
    predictions = predictions[valid].to(torch.int64)
    if targets.numel() == 0:
        return
    indices = targets * NUM_CLASSES + predictions
    confusion += torch.bincount(indices, minlength=NUM_CLASSES * NUM_CLASSES).reshape(NUM_CLASSES, NUM_CLASSES).cpu()
def evaluate(model, loader):
    model.eval()
    confusion = torch.zeros(NUM_CLASSES, NUM_CLASSES, dtype=torch.int64)
    loss_sum = 0.0
    batches = 0
    with torch.no_grad():
        for images, masks in loader:
            logits = model(images.to(DEVICE))
            if (masks != IGNORE_INDEX).sum().item() == 0:
                continue
            loss_sum += criterion(logits, masks.to(DEVICE)).item()
            predictions = logits.argmax(dim=1).cpu()
            update_confusion_matrix(confusion, predictions, masks)
            batches += 1
    if batches == 0 or confusion.sum().item() == 0:
        raise RuntimeError("Validation contains no valid pixels.")
    intersection = torch.diag(confusion).float()
    union = confusion.sum(1).float() + confusion.sum(0).float() - intersection
    valid_classes = confusion.sum(1) > 0
    iou = torch.full((NUM_CLASSES,), float("nan"))
    iou[valid_classes] = intersection[valid_classes] / union[valid_classes].clamp_min(1)
    pixel_accuracy = intersection.sum() / confusion.sum().float()
    return {
        "loss": loss_sum / batches,
        "per_class_iou": iou.tolist(),
        "mean_iou_valid_classes": torch.nanmean(iou).item(),
        "pixel_accuracy": pixel_accuracy.item(),
        "confusion_matrix": confusion,
    }
metrics = evaluate(model, validation_loader)
print({key: value for key, value in metrics.items() if key != "confusion_matrix"})

IoU is reported per class, with mean IoU computed only over classes having validation ground-truth pixels. Ignore pixels contribute to neither loss nor metrics. These values describe only the 128-image preliminary validation subset.

In [ ]:
import matplotlib.pyplot as plt

class_labels = ["soil", "bedrock", "sand", "big rock"]
fig, ax = plt.subplots(figsize=(6, 5))
image = ax.imshow(metrics["confusion_matrix"].numpy(), cmap="Blues")
ax.set_xlabel("Predicted class")
ax.set_ylabel("Ground-truth class")
ax.set_xticks(range(NUM_CLASSES), class_labels, rotation=30, ha="right")
ax.set_yticks(range(NUM_CLASSES), class_labels)
for row in range(NUM_CLASSES):
    for column in range(NUM_CLASSES):
        ax.text(column, row, int(metrics["confusion_matrix"][row, column]), ha="center", va="center")
fig.colorbar(image, ax=ax, label="pixels")
fig.tight_layout()
plt.show()


The confusion matrix excludes ID 255 and shows only valid terrain pixels. It should be interpreted as a baseline diagnostic, not as full AI4Mars performance.

In [ ]:
model.eval()
display_count = min(4, validation_images.shape[0])
with torch.no_grad():
    prediction_batch = model(validation_images[:display_count].to(DEVICE)).argmax(dim=1).cpu()
fig, axes = plt.subplots(3, display_count, figsize=(4 * display_count, 10), squeeze=False)
for column in range(display_count):
    axes[0, column].imshow(validation_images[column, 0], cmap="gray")
    axes[0, column].set_title("Input")
    axes[1, column].imshow(validation_masks[column], vmin=0, vmax=NUM_CLASSES - 1, cmap="viridis")
    axes[1, column].set_title("Ground truth")
    axes[2, column].imshow(prediction_batch[column], vmin=0, vmax=NUM_CLASSES - 1, cmap="viridis")
    axes[2, column].set_title("Prediction")
    for row in range(3):
        axes[row, column].axis("off")
plt.tight_layout()
plt.show()


## Baseline summary

This notebook defines a reproducible, lightweight U-Net baseline using 640 bounded streamed samples, a 512/128 preliminary image-level split, five epochs, and ignore-aware metrics. Execute it only in the intended Colab T4 runtime. The resulting losses and metrics must not be generalized to the complete AI4Mars dataset; a later audit must address duplicate images and rover-sequence grouping before production evaluation.